In [ ]:
import pandas as pd
from io import StringIO
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

# 1. CARGAR LOS DATOS

datos = """caso,conjunto,vida,enemigos,pociones,resultado_real
1,entrenamiento,14,5,3,1
2,entrenamiento,12,1,1,0
3,entrenamiento,34,4,2,1
4,entrenamiento,81,2,5,0
5,entrenamiento,50,5,3,1
6,entrenamiento,27,3,4,1
7,entrenamiento,8,5,2,1
8,entrenamiento,41,4,2,0
9,entrenamiento,6,4,4,1
10,entrenamiento,77,3,4,0
11,entrenamiento,71,4,3,1
12,entrenamiento,37,4,1,1
13,entrenamiento,64,2,3,0
14,entrenamiento,76,6,2,1
15,prueba,71,1,3,0
16,prueba,73,3,3,0
17,prueba,83,2,2,0
18,prueba,46,6,4,1
19,prueba,11,4,2,1
20,prueba,97,4,3,0
"""

df = pd.read_csv(StringIO(datos))


# 2. SEPARAR ENTRENAMIENTO Y PRUEBA

variables = ["vida", "enemigos", "pociones"]

entrenamiento = df[df["conjunto"] == "entrenamiento"]
prueba = df[df["conjunto"] == "prueba"]

X_train = entrenamiento[variables]
y_train = entrenamiento["resultado_real"]

X_test = prueba[variables]
y_test = prueba["resultado_real"]


# 3. ENTRENAR EL MODELO

modelo = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=2,
    random_state=42
)

modelo.fit(X_train, y_train)


# 4. DEFINIR LOS COSTOS Y EL UMBRAL

costo_FP = 2
costo_FN = 20

umbral = costo_FP / (costo_FP + costo_FN)


# 5. EVALUAR ENTRENAMIENTO Y PRUEBA

def evaluar(X, y, nombre):

    probabilidades = modelo.predict_proba(X)[:, 1]

    predicciones = (probabilidades >= umbral).astype(int)

    matriz = confusion_matrix(
        y, predicciones, labels=[0, 1]
    )

    TN, FP, FN, TP = matriz.ravel()

    exactitud = (TP + TN) / (TP + TN + FP + FN)

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0

    sensibilidad = TP / (TP + FN) if (TP + FN) > 0 else 0

    especificidad = TN / (TN + FP) if (TN + FP) > 0 else 0

    print("\n==============================")
    print(nombre)
    print("==============================")

    print("\nMatriz de confusión:")
    print(matriz)

    print("\nMétricas:")
    print("Exactitud:", round(exactitud, 2))
    print("Precisión:", round(precision, 2))
    print("Sensibilidad:", round(sensibilidad, 2))
    print("Especificidad:", round(especificidad, 2))


# Evaluar ambos conjuntos

evaluar(X_train, y_train, "ENTRENAMIENTO")

evaluar(X_test, y_test, "PRUEBA")


# 6. DEFINIR EL AGENTE RACIONAL

def agente(vida, enemigos, pociones):

    # Comprobar si hay recursos disponibles

    if pociones <= 0:
        return "NO CURAR: no quedan pociones."

    # Construir el caso nuevo

    nuevo_caso = pd.DataFrame(
        [[vida, enemigos, pociones]],
        columns=variables
    )

    # Estimar probabilidad de necesitar curación

    probabilidad = modelo.predict_proba(nuevo_caso)[0, 1]

    # Evaluar las consecuencias de la decisión

    if probabilidad >= umbral:
        return (
            f"SÍ, CURAR. "
            f"Probabilidad estimada: {probabilidad:.2f}"
        )

    else:
        return (
            f"NO CURAR. "
            f"Probabilidad estimada: {probabilidad:.2f}"
        )


# 7. PROBAR DOS CASOS INVENTADOS

print("\n==============================")
print("PRUEBAS DEL AGENTE")
print("==============================")

print("\nCASO INVENTADO 1")

print(agente(
    vida=15,
    enemigos=5,
    pociones=3
))

print("\nCASO INVENTADO 2")

print(agente(
    vida=80,
    enemigos=1,
    pociones=4
))


ENTRENAMIENTO

Matriz de confusión:
[[3 2]
 [0 9]]

Métricas:
Exactitud: 0.86
Precisión: 0.82
Sensibilidad: 1.0
Especificidad: 0.6

PRUEBA

Matriz de confusión:
[[2 2]
 [0 2]]

Métricas:
Exactitud: 0.67
Precisión: 0.5
Sensibilidad: 1.0
Especificidad: 0.5

PRUEBAS DEL AGENTE

CASO INVENTADO 1
SÍ, CURAR. Probabilidad estimada: 1.00

CASO INVENTADO 2
NO CURAR. Probabilidad estimada: 0.00
